In [1]:
import pandas as pd
import numpy as np
import torch as t
import torch.nn as nn
from torch.optim import Adam as Adam
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import gc
import os

In [2]:
'''
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)

# Later
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
'''

'\nfrom sklearn.preprocessing import StandardScaler\n\nscaler = StandardScaler()\nscaler.fit(X_train)\n\n# Later\nX_train_scaled = scaler.transform(X_train)\nX_test_scaled = scaler.transform(X_test)\n'

In [3]:
all_training_paths = [
    "data/5/exp05H20140926_10h50.csv",
    "data/5/exp05H20141001_10h05.csv",
    "data/5/exp05H20141003_15h00.csv",
    "data/5/exp05H20141008_15h30.csv",
    "data/5/exp05H20141010_10h38.csv",
    "data/5/exp05H20141015_12h50.csv",
    "data/5/exp05H20141022_11h45.csv",
    "data/5/exp05H20141023_16h20.csv",
    "data/5/exp05H20141029_14h20.csv",
]

In [4]:
def split_at_nans_vectorized(data: pd.DataFrame) -> list[pd.DataFrame]:
    """Splits data into contiguous chunks, breaking at any NaN values."""
    
    # 1. Create a boolean mask: True if any NaN exists in the row
    is_nan = data.isna().any(axis=1)
    
    # 2. Create unique group IDs. 
    # The cumulative sum increases by 1 every time it hits a NaN row.
    # This automatically assigns the same integer to contiguous blocks of valid data.
    block_ids = is_nan.cumsum()
    
    # 3. Filter out the actual NaN rows from both the data and the IDs
    valid_data = data[~is_nan]
    valid_block_ids = block_ids[~is_nan]
    
    # 4. Group by the IDs and extract the DataFrames as a list
    data_blocks = [group for _, group in valid_data.groupby(valid_block_ids)]
    
    return data_blocks

In [5]:
def create_sequences(data: np.ndarray, window_size = 30 ) -> tuple[np.ndarray, np.ndarray]:

    shape = data.shape

    X = np.zeros((shape[1], shape[0]-window_size, window_size))
    y = np.zeros((shape[1], shape[0]-window_size))

    for c in range(shape[1]):

        for i in range(len(data) - window_size):

            X[c,i] = data[i : i + window_size, c]
            
            y[c,i] = data[i + window_size, c]

    X = X.transpose(1, 2, 0)
    
    y = y.transpose(1, 0)

    return X, y

In [6]:
    def fit_scaler(list_of_paths: list[str]) -> StandardScaler:
        all_data = []

        scaler = StandardScaler()

        for path in list_of_paths:
            example_data = pd.read_csv(f'../{path}')
            blocks = split_at_nans_vectorized(example_data)

            for block in blocks:

                diffs = block.diff()

                diffs = diffs.iloc[1:]

                dfs = []

                for i in range(len(block.columns)//3):

                    v = np.sqrt(diffs[f'X{i+1}']**2 + diffs[f'Y{i+1}']**2)
                    av = diffs[f"H{i+1}"]
                    
                    dfs.append(v)
                    dfs.append(av)

                one_block = pd.concat(dfs, axis=1, ignore_index=True)

                all_data.append(one_block)
        
        train_data = pd.concat(all_data, axis=0, ignore_index=True)
        train_data.to_numpy()

        scaler = scaler.fit(train_data)

        return scaler

In [7]:
def convert_coords_to_v_and_av_scaled(coords_data: pd.DataFrame, scaler=None) -> np.array:
    if scaler is None:
        scaler = fit_scaler(all_training_paths)

    coords_diff = coords_data.diff().dropna()

    dfs = []
    num_objects = len(coords_diff.columns) // 3

    for i in range(num_objects):
        suffix = i + 1

        v = np.sqrt(coords_diff[f'X{suffix}']**2 + coords_diff[f'Y{suffix}']**2)

        av = coords_diff[f"H{suffix}"]
        
        dfs.append(v)
        dfs.append(av)

    df = pd.concat(dfs, axis=1, ignore_index=True) 

    df_np = df.to_numpy()

    df_scaled = scaler.transform(df_np)

    return df_scaled

In [8]:
def generate_scaled_sequences(list_of_paths: list[str], scaler: StandardScaler, window_size: int = 50):
    x_all = []
    y_all = []

    for path in list_of_paths:
        example_data = pd.read_csv(f'../{path}')

        blocks = split_at_nans_vectorized(example_data)

        for block in blocks:
            one_block_np_scaled = convert_coords_to_v_and_av_scaled(block, scaler=scaler)
            
            if len(one_block_np_scaled) > window_size:
                X, y = create_sequences(one_block_np_scaled, window_size)
            
                x_all.append(X)
                y_all.append(y)
    if not x_all:
        print("Warning: No sequences were generated. Check window_size or data quality.")
        return np.array([]), np.array([])

    X_full = np.concatenate(x_all, axis=0)
    y_full = np.concatenate(y_all, axis=0)
    
    return X_full, y_full

fitted_scaler = fit_scaler(all_training_paths)
X_full, y_full = generate_scaled_sequences(all_training_paths, scaler=fitted_scaler)

print(f"Final X shape: {X_full.shape}")
print(f"Final y shape: {y_full.shape}")

Final X shape: (1407990, 50, 10)
Final y shape: (1407990, 10)


In [26]:
y_one_fish = y_full[:,:2]

X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_one_fish,
    test_size=0.2,
    random_state=42
)
print(f"Final X_train shape: {X_train.shape}")
print(f"Final y_train shape: {y_train.shape}")
print(f"Final X_val shape: {X_val.shape}")
print(f"Final y_val shape: {y_val.shape}")
print(f"X_train has NaNs: {np.isnan(X_train).any()}")
print(f"y_train has NaNs: {np.isnan(y_train).any()}")

Final X_train shape: (1126392, 50, 10)
Final y_train shape: (1126392, 2)
Final X_val shape: (281598, 50, 10)
Final y_val shape: (281598, 2)
X_train has NaNs: False
y_train has NaNs: False


In [27]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)

In [29]:
print(f"X_train shape: {X_train_flat.shape}, type : {type(X_train)}")
print(f"y_train shape: {y_train.shape}, type : {type(y_train)}")
print(f"X_val shape: {X_val_flat.shape}, type : {type(X_val)}")
print(f"y_val shape: {y_val.shape}, type : {type(y_val)}")

X_train shape: (1126392, 500), type : <class 'numpy.ndarray'>
y_train shape: (1126392, 2), type : <class 'numpy.ndarray'>
X_val shape: (281598, 500), type : <class 'numpy.ndarray'>
y_val shape: (281598, 2), type : <class 'numpy.ndarray'>


In [12]:
class MyLinear(nn.Module):
    def __init__(self, in_features, out_features):
        '''
        For any layer there is a weight between all the in features and out features
        There is also a bias, initially these are all normally distributed
        '''
        super().__init__()
        
        #tune down starting weigths and biases bc first training loop gave massive loss
        self.weights = nn.Parameter(t.randn(in_features, out_features, requires_grad=True)*0.01) 
        self.biases  = nn.Parameter(t.zeros(out_features, requires_grad=True))

    def forward(self, x):
        '''
        Here, x is basically the input layer values and it returns the output layer values
        '''
        # print(f"self.weights.shape = {self.weights.shape}")
        # print(f"self.biases.shape = {self.biases.shape}")
        # print(f"x.shape = {x.shape}")
        return x @ self.weights + self.biases

In [13]:
class MyNet(nn.Module):
    def __init__(self, in_features, hidden_features1, hidden_features2, out_features):
        super().__init__()
        '''
        This nn has one hidden layer between input and output
        '''
        self.layer1 = MyLinear(in_features, hidden_features1)
        self.layer2 = MyLinear(hidden_features1, hidden_features2)
        self.layer3 = MyLinear(hidden_features2, out_features)

        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        '''
        Whne calling self.layer 1, this has already been initialised so uses the forward function in MyLinear.
        x is the input layer so this passes the input through the nn, applying the weights and biases to get the output.
        '''
        x = x.reshape(x.shape[0], -1) # (batch_size, 30, 4) -> (batch_size, 120)

        x = self.layer1(x)
        x = t.relu(x)
        x = self.dropout(x)

        x = self.layer2(x)
        x = t.relu(x)
        x = self.dropout(x)
        
        x = self.layer3(x)
        return x

In [14]:
def loss(predictions, targets):
    # Calculates MSE
    return t.mean((predictions - targets) ** 2)

In [ ]:
'''
X_train_t = t.tensor(X_train, dtype=t.float32)
y_train_t = t.tensor(y_train, dtype=t.float32)
X_val_t = t.tensor(X_val, dtype=t.float32)
y_val_t = t.tensor(y_val, dtype=t.float32)

learning_rate = 0.001
epochs = 100
mynet = MyNet(120, 256, 128, 4)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

optimizer = Adam(mynet.parameters(), lr=learning_rate, weight_decay=1e-4)

for epoch in range(epochs):
    mynet.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        
        # Forward pass: pass in X_train_t, not X!
        prediction = mynet(batch_X) 
        
        # Calculate loss: compare predictions to y_train_t
        L = loss(prediction, batch_y)
        
        # Backward pass
        L.backward()
        
        # Update weights
        optimizer.step()
        
    if epoch % 10 == 0:
        mynet.eval()
        with t.no_grad():
            full_train_pred = mynet(X_train_t)
            train_loss = loss(full_train_pred, y_train_t)

            val_prediction = mynet(X_val_t)
            val_loss = loss(val_prediction, y_val_t)
        print(f"Epoch {epoch} | Train Loss: {train_loss.item():.4f} | Val Loss: {val_loss.item():.4f}")
'''

In [15]:
class MyLSTMNet(nn.Module):
    def __init__(self, input_features, hidden_size, out_features):
        super().__init__()
        # batch_first=True tells PyTorch your shape is (batch, seq, feature)
        self.lstm = nn.LSTM(
            input_size=input_features, 
            hidden_size=hidden_size, 
            num_layers=2,
            dropout=0.2,
            batch_first=True 
        )
        self.dropout = nn.Dropout(p=0.2) #Regularization (reduce overfitting)
        
        self.fc = nn.Linear(hidden_size, out_features) #The Output Layer

    def forward(self, x):
        
        # The LSTM outputs two things:
        # lstm_out: The hidden states for ALL 30 time steps
        # (hn, cn): The final hidden and cell states at the very end
        lstm_out, (hn, cn) = self.lstm(x)

        # [:, -1, :] means -> [all batches, the last time step, all hidden features]
        last_time_step = lstm_out[:, -1, :] 
        
        # Apply dropout and pass through the final linear layer
        out = self.dropout(last_time_step)

        predictions = self.fc(out)
        
        return predictions

The LSTM (long short term memory) code is 90% gemini, will do an explanation of how it works in presentation. 

In [16]:
class FishSequenceDataset(Dataset):
    def __init__(self, X_data, y_data):
        # Store the raw NumPy arrays directly. 
        # No massive upfront tensor conversions!
        self.X = X_data
        self.y = y_data

    def __len__(self):
        # PyTorch needs to know the total number of sequences to calculate epochs
        return len(self.X)

    def __getitem__(self, idx):
        # This is called internally by the DataLoader.
        # We convert to PyTorch tensors piece-by-piece.
        x_seq = t.tensor(self.X[idx], dtype=t.float32)
        y_target = t.tensor(self.y[idx], dtype=t.float32)
        
        return x_seq, y_target

In [20]:
X_train = X_train.astype('float32')
y_train = y_train.astype('float32')
X_val = X_val.astype('float32')
y_val = y_val.astype('float32')

In [30]:
X_train = X_train_flat.astype('float32')
y_train = y_train.astype('float32')
X_val = X_val_flat.astype('float32')
y_val = y_val.astype('float32')

In [31]:
print(f"X_train shape: {X_train.shape}, type : {type(X_train)}")
print(f"y_train shape: {y_train.shape}, type : {type(y_train)}")
print(f"X_val shape: {X_val.shape}, type : {type(X_val)}")
print(f"y_val shape: {y_val.shape}, type : {type(y_val)}")

X_train shape: (1126392, 500), type : <class 'numpy.ndarray'>
y_train shape: (1126392, 2), type : <class 'numpy.ndarray'>
X_val shape: (281598, 500), type : <class 'numpy.ndarray'>
y_val shape: (281598, 2), type : <class 'numpy.ndarray'>


In [32]:
train_set = FishSequenceDataset(X_train, y_train)
val_set = FishSequenceDataset(X_val, y_val)

In [22]:
print(f"X_train has NaNs: {np.isnan(X_val).any()}")
print(f"y_train has NaNs: {np.isnan(y_train).any()}")

X_train has NaNs: False
y_train has NaNs: False


In [ ]:
'''
gc.collect()

device = t.device("mps" if t.backends.mps.is_available() else "cpu")

model = MyLSTMNet(input_features=10, hidden_size=32, out_features=2)
model = model.to(device)

epochs = 200
batch_size = 32

optim = t.optim.SGD(model.parameters(), lr=3e-4)
loss_func = t.nn.MSELoss()

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

for epoch in range(epochs): 
    model.train()
    total_train_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        
        optim.zero_grad()

        y_pred = model(x)

        train_loss = loss_func(y_pred,y)

        total_train_loss += train_loss.item()

        train_loss.backward()
        
        optim.step()

    if epoch % 20 == 0:
        total_val_loss = 0
        model.eval()

        with t.no_grad():
            for x_val, y_val in val_loader:
                x_val = x_val.to(device)
                y_val = y_val.to(device)

                y_val_pred = model(x_val)

                val_loss = loss_func(y_val_pred, y_val)

                total_val_loss += val_loss.item()

        avg_train_loss = total_train_loss/len(train_loader)
        avg_val_loss = total_val_loss/len(val_loader)

        print(f"Epoch {epoch}: Avg Train Loss: {avg_train_loss:.4f} | Avg Val Loss: {avg_val_loss:.4f}")
'''

From SDG, batch size = 32, time to run == 10003 min, model = MyLSTMNet(input_features=10, hidden_size=32, out_features=2)
Epoch 0: Avg Train Loss: 0.7947 | Avg Val Loss: 0.6988
Epoch 20: Avg Train Loss: 0.6123 | Avg Val Loss: 0.5740
Epoch 40: Avg Train Loss: 0.5240 | Avg Val Loss: 0.4779 n
Epoch 60: Avg Train Loss: 0.4708 | Avg Val Loss: 0.4255
Epoch 80: Avg Train Loss: 0.4444 | Avg Val Loss: 0.3990
Epoch 100: Avg Train Loss: 0.4287 | Avg Val Loss: 0.3838
Epoch 120: Avg Train Loss: 0.4186 | Avg Val Loss: 0.3741
Epoch 140: Avg Train Loss: 0.4120 | Avg Val Loss: 0.3672
Epoch 160: Avg Train Loss: 0.4072 | Avg Val Loss: 0.3623

In [42]:
'''t.save(model.state_dict(), "SGD_fish_model.pth")'''

In [33]:
checkpoint_path = 'model_checkpoint_linear_2.pth'

gc.collect()

device = t.device("mps" if t.backends.mps.is_available() else "cpu")

#model = MyLSTMNet(input_features=10, hidden_size=128, out_features=2)
model = MyNet(500, 512, 256, 2)

model = model.to(device)

epochs = 200
batch_size = 16
optim = t.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
loss_func = t.nn.MSELoss()

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)

start_epoch = 0
best_val_loss = float('inf')

if os.path.exists(checkpoint_path):
    checkpoint = t.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optim.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))
    print(f"--- Resuming training from Epoch {start_epoch} ---")

for epoch in range(start_epoch, epochs): 
    model.train()
    total_train_loss = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        
        optim.zero_grad()

        y_pred = model(x)

        train_loss = loss_func(y_pred,y)

        total_train_loss += train_loss.item()

        train_loss.backward()
        
        optim.step()

    if epoch % 10 == 0:
        total_val_loss = 0
        model.eval()

        with t.no_grad():
            for x_val, y_val in val_loader:
                x_val = x_val.to(device)
                y_val = y_val.to(device)

                y_val_pred = model(x_val)

                val_loss = loss_func(y_val_pred, y_val)

                total_val_loss += val_loss.item()

        avg_train_loss = total_train_loss/len(train_loader)
        avg_val_loss = total_val_loss/len(val_loader)

        print(f"Epoch {epoch}: Avg Train Loss: {avg_train_loss:.4f} | Avg Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            t.save(model.state_dict(), "best_model_linear_2.pth")
            print(f"New best model saved at epoch {epoch}")

    t.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'best_val_loss': best_val_loss,
    }, checkpoint_path)
    
    print(f"Epoch {epoch} complete. Checkpoint saved.")

Epoch 0: Avg Train Loss: 0.5412 | Avg Val Loss: 0.5453
New best model saved at epoch 0
Epoch 0 complete. Checkpoint saved.
Epoch 1 complete. Checkpoint saved.
Epoch 2 complete. Checkpoint saved.
Epoch 3 complete. Checkpoint saved.
Epoch 4 complete. Checkpoint saved.
Epoch 5 complete. Checkpoint saved.
Epoch 6 complete. Checkpoint saved.
Epoch 7 complete. Checkpoint saved.
Epoch 8 complete. Checkpoint saved.
Epoch 9 complete. Checkpoint saved.
Epoch 10: Avg Train Loss: 0.5355 | Avg Val Loss: 0.4056
New best model saved at epoch 10
Epoch 10 complete. Checkpoint saved.
Epoch 11 complete. Checkpoint saved.
Epoch 12 complete. Checkpoint saved.
Epoch 13 complete. Checkpoint saved.
Epoch 14 complete. Checkpoint saved.
Epoch 15 complete. Checkpoint saved.
Epoch 16 complete. Checkpoint saved.


KeyboardInterrupt: 

SLTM - 128 hidden units, 2 layers
Epoch 0: Avg Train Loss: 0.3324 | Avg Val Loss: 0.2828
New best model saved at epoch 0
Epoch 0 complete. Checkpoint saved.
Epoch 1 complete. Checkpoint saved.
Epoch 2 complete. Checkpoint saved.
Epoch 3 complete. Checkpoint saved.
Epoch 4 complete. Checkpoint saved.
Epoch 5 complete. Checkpoint saved.
Epoch 6 complete. Checkpoint saved.
Epoch 7 complete. Checkpoint saved.
Epoch 8 complete. Checkpoint saved.
Epoch 9 complete. Checkpoint saved.
Epoch 10: Avg Train Loss: 0.3029 | Avg Val Loss: 0.2770
New best model saved at epoch 10
Epoch 10 complete. Checkpoint saved.
Epoch 11 complete. Checkpoint saved.
Epoch 12 complete. Checkpoint saved.
Epoch 13 complete. Checkpoint saved.
Epoch 14 complete. Checkpoint saved.
Epoch 15 complete. Checkpoint saved.
Epoch 16 complete. Checkpoint saved.
Epoch 17 complete. Checkpoint saved.
Epoch 18 complete. Checkpoint saved.
Epoch 19 complete. Checkpoint saved.
Epoch 20: Avg Train Loss: 0.3021 | Avg Val Loss: 0.2718
New best model saved at epoch 20
Epoch 20 complete. Checkpoint saved.
Epoch 21 complete. Checkpoint saved.
Epoch 22 complete. Checkpoint saved.
Epoch 23 complete. Checkpoint saved.
Epoch 24 complete. Checkpoint saved.
Epoch 25 complete. Checkpoint saved.
Epoch 26 complete. Checkpoint saved.
Epoch 27 complete. Checkpoint saved.
Epoch 28 complete. Checkpoint saved.
Epoch 29 complete. Checkpoint saved.
Epoch 30: Avg Train Loss: 0.3023 | Avg Val Loss: 0.2718
Epoch 30 complete. Checkpoint saved.
Epoch 31 complete. Checkpoint saved.
Epoch 32 complete. Checkpoint saved.
Epoch 33 complete. Checkpoint saved.
Epoch 34 complete. Checkpoint saved.
Epoch 35 complete. Checkpoint saved.
Epoch 36 complete. Checkpoint saved.
Epoch 37 complete. Checkpoint saved.
Epoch 38 complete. Checkpoint saved.
Epoch 39 complete. Checkpoint saved.
Epoch 40: Avg Train Loss: 0.3022 | Avg Val Loss: 0.2708
New best model saved at epoch 40
Epoch 40 complete. Checkpoint saved.
Epoch 41 complete. Checkpoint saved.
Epoch 42 complete. Checkpoint saved.


optim = t.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
model = MyNet(500, 512, 256, 2)    

Epoch 0: Avg Train Loss: 0.5412 | Avg Val Loss: 0.5453
New best model saved at epoch 0
Epoch 0 complete. Checkpoint saved.
Epoch 1 complete. Checkpoint saved.
Epoch 2 complete. Checkpoint saved.
Epoch 3 complete. Checkpoint saved.
Epoch 4 complete. Checkpoint saved.
Epoch 5 complete. Checkpoint saved.
Epoch 6 complete. Checkpoint saved.
Epoch 7 complete. Checkpoint saved.
Epoch 8 complete. Checkpoint saved.
Epoch 9 complete. Checkpoint saved.
Epoch 10: Avg Train Loss: 0.5355 | Avg Val Loss: 0.4056
New best model saved at epoch 10
Epoch 10 complete. Checkpoint saved.
Epoch 11 complete. Checkpoint saved.
Epoch 12 complete. Checkpoint saved.
Epoch 13 complete. Checkpoint saved.
Epoch 14 complete. Checkpoint saved.
Epoch 15 complete. Checkpoint saved.
Epoch 16 complete. Checkpoint saved.


In [63]:
def convert_v_and_av_to_cords_unscaled(real_path, predicted_path, scaler=None):
    prediction_full = np.zeros(real_path.shape)
    
    if scaler is None:
        scaler = fit_scaler(all_training_paths)

    predicted_motion_unscaled = scaler.inverse_transform(predicted_path)

    for i in range(real_path.shape[1] // 3):
        x_start = real_path[0, 3*i]
        y_start = real_path[0, 1 + 3*i]
        h_start = real_path[0, 2 + 3*i] % (2 * np.pi)

        av_cumsum = np.cumsum(predicted_motion_unscaled[:, 1 + 2*i])
        predicted_headings = (h_start + av_cumsum) % (2 * np.pi)

        delta_x = predicted_motion_unscaled[:, 0 + 2*i] * np.cos(predicted_headings) 
        delta_y = predicted_motion_unscaled[:, 0 + 2*i] * np.sin(predicted_headings)

        predicted_x = x_start + np.cumsum(delta_x)
        predicted_y = y_start + np.cumsum(delta_y)

        prediction_full[:, 3*i] = predicted_x
        prediction_full[:, 1 + 3*i] = predicted_y
        prediction_full[:, 2 + 3*i] = predicted_headings

    return real_path, prediction_full

In [67]:
model_SGD = MyLSTMNet(input_features=10, hidden_size=32, out_features=2)
model_AdamW = MyLSTMNet(input_features=10, hidden_size=128, out_features=2)

def predict_path(model:nn.Module, wieghts_path:str, test_path:str, path_range: list[int,int], window_size = 50):
    model.load_state_dict(t.load(wieghts_path))
    model.eval()

    start_idx, finish_idx = path_range[0], path_range[1]

    test_path = pd.read_csv(f"../{test_path}")

    real_path = test_path.iloc[start_idx:finish_idx]

    predicted_path = convert_v_and_av_to_cords_unscaled(real_path, predict_path) #Retuns numpy array

    for i in range(finish_idx-start_idx-window_size):
        x = predicted_path.iloc[i:window_size + i]

        x_tensor = t.tensor(x, dtype=t.float32)

        predictions = model(x_tensor)

        preds_array = predictions.detach().cpu().numpy()

        v_pred = preds_array[0][0]
        av_pred = preds_array[0][1]

        predicted_path[0,window_size + i +1] = v_pred
        predicted_path[1,window_size + i +1] = av_pred
    
    return real_path, predicted_path

In [64]:
def plot_real_vs_predicted_paths(real_path, prediction_full):
    fig1, (ax1, ax2) = plt.subplots(1,2, figsize = (8,13))
    for i in range(5):
        ax1.plot(real_path[:, 3*i], real_path[:, 1 + 3*i], label = f"Fish {i+1}")
        ax2.plot(prediction_full[:, 3*i], prediction_full[:, 1 + 3*i], label = f"Fish {i+1}")

    ax1.legend()
    ax2.legend()

    ax1.set_aspect('equal')
    ax2.set_aspect('equal')

    ax1.set_title("Real Path")
    ax2.set_title("Predictied path for fish 1")

    plt.show()

In [ ]:
/Users/thomasleaver/Documents/Coding/MDM2_2/mdm2_project2/data/5/exp05H20141030_11h15.csv

In [ ]:
real_path, predicted_path = predict_path(model = model_SGD, weights_path = "SGD_fish_model.pth", test_path = "data/5/exp05H20141030_11h15.csv",path_range = [10200,10500])
real_path, predicted_path = convert_v_and_av_to_cords_unscaled(real_path, predicted_path)
plot_real_vs_predicted_paths(real_path, predicted_path)

In [ ]:
example_data = pd.read_csv("../data/5/exp05H20141030_11h15.csv")

def plot_fish_paths(experement:pd.DataFrame) -> None :
    n_fish = experement.shape[1]//3

    for i in range(0,n_fish*3,3) :
        plt.plot(experement.iloc[:,i],experement.iloc[:,i+1],label=f'fish {1 + i//3}')

    plt.legend()
    plt.show('plots/paths.pdf')

plot_fish_paths(example_data.loc[10200:10500])